# 01 · Exploration et préparation (IEEE-CIS)
Notebook de support : mêmes fonctions que le pipeline (`src/`), pour explorer pas à pas.

In [ ]:
import sys; sys.path.insert(0, '..')
import pandas as pd, numpy as np, matplotlib.pyplot as plt
from src import features as F
from src.download import ensure_raw_data
ensure_raw_data()   # téléchargement Kaggle automatique si absent

In [ ]:
df = F.load_raw(max_rows=0).sort_values('TransactionDT', kind='stable').reset_index(drop=True)
print(df.shape, 'taux de fraude : %.2f %%' % (100*df.isFraud.mean()))

## Variables temporelles et comportementales (passé uniquement)

In [ ]:
df = F.add_time_features(df.copy()); df = F.add_behavior_features(df)
df[F.BEHAV_FEATS].describe().T

In [ ]:
g = df.groupby(df.tx_count_1h.clip(upper=4)).isFraud.mean()*100
g.plot.bar(ylabel='Taux de fraude (%)', title='Fraude selon le nombre d\'achats sur 1 h'); plt.show()

## Découpage chronologique 60/20/20

In [ ]:
from src import config as C
n = len(df); n_tr, n_va = C.split_points(n)
for name, part in {'train': df.iloc[:n_tr], 'validation': df.iloc[n_tr:n_va], 'test': df.iloc[n_va:]}.items():
    print(name, len(part), '%.2f %% de fraude' % (100*part.isFraud.mean()))

## Construire tous les artefacts
Dans un terminal : `python -m src.pipeline` (ou `--max-rows 150000` pour aller plus vite).